1. LOADING DATASET

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('clean_credit_risk_data.csv')

In [3]:
df.head()

,id,loan_amnt,term,int_rate,installment,grade,sub_grade,annual_inc,loan_status,dti,fico_range_low,fico_range_high,revol_util,pub_rec_bankruptcies,purpose,addr_state,utilization_diff,high_util_risk,high_dti_risk,total_risk_score
0,68407277,3600.0,36 months,13.99,123.03,C,C4,55000.0,Fully Paid,5.91,675.0,679.0,29.7,0.0,debt_consolidation,PA,17.7,0,0,0
1,68355089,24700.0,36 months,11.99,820.28,C,C1,65000.0,Fully Paid,16.06,715.0,719.0,19.2,0.0,small_business,SD,7.2,0,0,0
2,68341763,20000.0,60 months,10.78,432.66,B,B4,63000.0,Fully Paid,10.78,695.0,699.0,56.2,0.0,home_improvement,IL,44.2,1,0,1
3,66310712,35000.0,60 months,14.85,829.90,C,C5,110000.0,Current,17.06,785.0,789.0,11.6,0.0,debt_consolidation,NJ,-0.4,0,0,0
4,68476807,10400.0,60 months,22.45,289.91,F,F1,104433.0,Fully Paid,25.37,695.0,699.0,64.5,0.0,major_purchase,PA,52.5,1,1,2


2. Create Target Variable

In [4]:
df['target'] = df['loan_status'].apply(
    lambda x: 1 if x in [
        'Charged Off',
        'Default',
        'Late (31-120 days)'
    ] else 0
)

df['target'].value_counts()

target
0    81925
1    18036
Name: count, dtype: int64

3. Select Important Features

In [5]:
numerical_cols = [
    'annual_inc',
    'fico_range_low',
    'dti',
    'loan_amnt',
    'int_rate',
    'installment',
    'total_risk_score'
]

categorical_cols = [
    'grade',
    'purpose',
    'term'
]

4. Encode Categorical Columns + Create X and y

In [6]:
X = df[numerical_cols]

encoded_df = pd.get_dummies(
    df[categorical_cols],
    drop_first=True
)

X = pd.concat([X, encoded_df], axis=1)

y = df['target']

X.head()

,annual_inc,fico_range_low,dti,loan_amnt,int_rate,installment,total_risk_score,grade_B,grade_C,grade_D,...,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,term_ 60 months
0,55000.0,675.0,5.91,3600.0,13.99,123.03,0,False,True,False,...,False,False,False,False,False,False,False,False,False,False
1,65000.0,715.0,16.06,24700.0,11.99,820.28,0,False,True,False,...,False,False,False,False,False,False,False,True,False,False
2,63000.0,695.0,10.78,20000.0,10.78,432.66,1,True,False,False,...,True,False,False,False,False,False,False,False,False,True
3,110000.0,785.0,17.06,35000.0,14.85,829.90,0,False,True,False,...,False,False,False,False,False,False,False,False,False,True
4,104433.0,695.0,25.37,10400.0,22.45,289.91,2,False,False,False,...,False,False,True,False,False,False,False,False,False,True


5.Train-Test Split

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

6. Train Balanced Logistic Regression Model

In [10]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=3000,
    class_weight='balanced',
    solver='liblinear'
)

model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=3000, solver='liblinear')

7. Generate Predictions

In [11]:
y_pred = model.predict(X_test)

y_pred[:10]

array([1, 1, 0, 1, 0, 0, 1, 1, 0, 0], dtype=int64)

8. FINAL EVALUATION - Balanced Logistic Regression Model

In [12]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

print(confusion_matrix(y_test, y_pred))

Accuracy: 0.6491772120242085
              precision    recall  f1-score   support

           0       0.88      0.66      0.76     16412
           1       0.28      0.61      0.38      3581

    accuracy                           0.65     19993
   macro avg       0.58      0.63      0.57     19993
weighted avg       0.78      0.65      0.69     19993

[[10809  5603]
 [ 1411  2170]]
